<a href="https://colab.research.google.com/github/NotApel/Read-Me/blob/main/Smart_Meeting_Assistant_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Meeting Assistant


## Step 1: Install libraries

In [1]:
!pip install -q transformers sentencepiece gradio

## Step 2: Load the pre-trained AI models
The first run downloads the models (about 1 GB in total), so it takes a few minutes.

In [2]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device.upper())

def load_model(name):
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSeq2SeqLM.from_pretrained(name).to(device)
    model.eval()
    return tokenizer, model

def generate_text(tokenizer, model, text, max_new_tokens, min_new_tokens=0, max_input=1024, num_beams=4):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input).to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                min_new_tokens=min_new_tokens, num_beams=num_beams, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

# Model 1: summarisation
sum_tok, sum_model = load_model("sshleifer/distilbart-cnn-12-6")

# Model 2: instruction-following text generation (used to rewrite action items)
task_tok, task_model = load_model("google/flan-t5-base")

print("Models loaded.")

Running on: CPU


config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

... (13 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Models loaded.


## Step 3: Sample meeting transcript (for testing and for your demo)

In [5]:
SAMPLE = """Aina: Good morning everyone. Today we need to finalise our SDT 2143 hackathon project before the deadline on Friday.
Hafiz: I finished the first version of the notebook yesterday. The summariser works well, but the interface still looks plain.
Aina: Great. Hafiz, please improve the Gradio interface and add a time-saved counter by Wednesday.
Mei Ling: I tested the tool with three different transcripts. It struggles when the transcript is very long, so we should split the text into smaller chunks.
Aina: Good point. Mei Ling will implement the chunking function and test it with at least five transcripts by Thursday.
Hafiz: We also have to decide which theme to present. I think productivity is the best fit because everyone has long meetings.
Aina: Agreed, we will go with AI for Productivity. I will write the project proposal and the introduction section tonight.
Mei Ling: Someone needs to prepare the presentation slides and record the demo video.
Aina: I'll prepare the slides on Thursday evening, and Hafiz will record the demo video on Friday morning.
Hafiz: Please remember to take screenshots of every step for the reflection report.
Aina: Yes. Everyone must send their screenshots to the shared folder by Thursday. Okay, that is all. Thank you everyone."""

## Step 4: AI functions

In [3]:
def split_into_chunks(text, max_words=350):
    """Split a long transcript into chunks the model can handle."""
    sentences = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    chunks, current, count = [], [], 0
    for s in sentences:
        n = len(s.split())
        if count + n > max_words and current:
            chunks.append(" ".join(current))
            current, count = [], 0
        current.append(s)
        count += n
    if current:
        chunks.append(" ".join(current))
    return chunks


def summarize_meeting(text):
    """AI summary: summarise each chunk, then join the pieces."""
    parts = []
    for chunk in split_into_chunks(text):
        n = len(chunk.split())
        if n < 25:
            parts.append(chunk)
            continue
        max_len = max(25, min(90, int(n * 0.5)))
        min_len = max(10, int(max_len * 0.3))
        parts.append(generate_text(sum_tok, sum_model, chunk,
                                   max_new_tokens=max_len, min_new_tokens=min_len))
    return " ".join(parts)


# Words/phrases that usually signal a task or commitment
ACTION_PATTERNS = [
    r"\bwill\b", r"\bneeds? to\b", r"\bshould\b", r"\bmust\b", r"\bhave to\b",
    r"\bplease\b", r"\bdeadline\b", r"\bfollow up\b", r"\bassign",
    r"\b\w+'ll\b", r"\bby (monday|tuesday|wednesday|thursday|friday|saturday|sunday|tomorrow|next week)\b",
]


def extract_action_items(text, max_items=8):
    """Find task sentences, then use the AI model to rewrite each as a short to-do."""
    sentences = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    candidates = [s.strip() for s in sentences
                  if s.strip() and any(re.search(p, s, re.I) for p in ACTION_PATTERNS)]
    items = []
    for s in candidates[:max_items]:
        prompt = f"Rewrite this as a short to-do task that says who must do what and by when: {s}"
        out = generate_text(task_tok, task_model, prompt, max_new_tokens=40,
                            max_input=512, num_beams=2)
        item = out if out else s
        if item not in items:
            items.append(item)
    return items


def reading_time(text, wpm=200):
    return len(text.split()) / wpm  # minutes


def run_assistant(text):
    if not text or len(text.split()) < 30:
        return "Please paste a longer transcript (at least 30 words).", "", ""
    summary = summarize_meeting(text)
    items = extract_action_items(text)
    action_text = "\n".join(f"{i}. {a}" for i, a in enumerate(items, 1)) or "No action items detected."
    saved = max(0.0, reading_time(text) - reading_time(summary))
    stats = (f"Original: {len(text.split())} words | Summary: {len(summary.split())} words\n"
             f"Estimated reading time saved: {saved:.1f} minutes")
    return summary, action_text, stats

## Step 5: Test it (take a screenshot of this output for your report)

In [6]:
summary, actions, stats = run_assistant(SAMPLE)
print("SUMMARY\n", summary, "\n")
print("ACTION ITEMS\n", actions, "\n")
print(stats)

[transformers] Both `max_new_tokens` (=90) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=27) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SUMMARY
 SDT 2143 hackathon project needs to be finalised before Friday deadline . Aina will write the project proposal and the introduction section tonight . Hafiz will record the demo video on Friday morning . Mei Ling will implement chunking function and test it with at least five transcripts by Thursday . 

ACTION ITEMS
 1. finalise our SDT 2143 project
2. How can I save time in Gradio?
3. How to split a transcript into smaller chunks?
4. How can I test the chunking function?
5. What should we do next?
6. Aina: We will go with AI for Productivity.
7. Who must write the project proposal and the introduction section tonight?
8. Who needs to prepare the presentation slides and record the demo video? 

Original: 208 words | Summary: 51 words
Estimated reading time saved: 0.8 minutes


## Step 6: Launch the web interface (use this for your demo)
Click the public link that appears to open the app.

In [7]:
import gradio as gr

demo = gr.Interface(
    fn=run_assistant,
    inputs=gr.Textbox(lines=14, label="Paste meeting transcript or notes here"),
    outputs=[
        gr.Textbox(label="AI Summary", lines=6),
        gr.Textbox(label="Action Items", lines=8),
        gr.Textbox(label="Productivity Impact", lines=2),
    ],
    title="Smart Meeting Assistant",
    description="AI for Productivity: turn long meeting notes into a short summary and a clear to-do list.",
    examples=[[SAMPLE]],
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://40ed721e8069f9bc45.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
